# Calculating the Accuracy & F1-Score:

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import evaluate

MODEL_PATH = "saved_model"
TOKENIZER_SOURCE = "bert-base-uncased"  # fresh tokenizer, not from MODEL_PATH

# 1. Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
if torch.cuda.is_available():
    model.to("cuda")

# 2. Load and preprocess test set (same steps as training)
dataset = load_dataset("sh0416/ag_news")

def preprocess(batch):
    text = [t + " " + d for t, d in zip(batch["title"], batch["description"])]
    tokenized = tokenizer(text, truncation=True, padding="max_length", max_length=128)
    tokenized["labels"] = [label - 1 for label in batch["label"]]
    return tokenized

tokenized_test = dataset["test"].map(preprocess, batched=True, remove_columns=["title", "description", "label"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 3. Get predictions for the whole test set, in small batches to avoid
#    running out of memory (running everything in one pass can need several GB)
test_data = tokenized_test[:]  # materializes into a dict of real tensors

batch_size = 8
all_preds = []

with torch.no_grad():
    for i in range(0, len(test_data["input_ids"]), batch_size):
        batch_inputs = {
            "input_ids": test_data["input_ids"][i : i + batch_size].to(model.device),
            "attention_mask": test_data["attention_mask"][i : i + batch_size].to(model.device),
        }
        logits = model(**batch_inputs).logits
        batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(batch_preds)

        if i % (batch_size * 20) == 0:
            print(f"Processed {i}/{len(test_data['input_ids'])} examples...")

preds = np.array(all_preds)
labels = test_data["labels"].numpy()

# 4. Compute accuracy and F1
accuracy = evaluate.load("accuracy").compute(predictions=preds, references=labels)
f1 = evaluate.load("f1").compute(predictions=preds, references=labels, average="macro")

print("Accuracy:", accuracy["accuracy"])
print("F1 (macro):", f1["f1"])

c:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2788.42it/s]


Processed 0/7600 examples...
Processed 160/7600 examples...
Processed 320/7600 examples...
Processed 480/7600 examples...
Processed 640/7600 examples...
Processed 800/7600 examples...
Processed 960/7600 examples...
Processed 1120/7600 examples...
Processed 1280/7600 examples...
Processed 1440/7600 examples...
Processed 1600/7600 examples...
Processed 1760/7600 examples...
Processed 1920/7600 examples...
Processed 2080/7600 examples...
Processed 2240/7600 examples...
Processed 2400/7600 examples...
Processed 2560/7600 examples...
Processed 2720/7600 examples...
Processed 2880/7600 examples...
Processed 3040/7600 examples...
Processed 3200/7600 examples...
Processed 3360/7600 examples...
Processed 3520/7600 examples...
Processed 3680/7600 examples...
Processed 3840/7600 examples...
Processed 4000/7600 examples...
Processed 4160/7600 examples...
Processed 4320/7600 examples...
Processed 4480/7600 examples...
Processed 4640/7600 examples...
Processed 4800/7600 examples...
Processed 4960/76

Accuracy: 0.9461842105263157
F1 (macro): 0.9461827045867581


## Final Results:
#### - Accuracy : 94%
#### - F1 (macro) : 94%